# MODIS (OBPG) vs Copernicus Marine: Daily SST + Chl-a comparison (FI study area)

This notebook provides an **end-to-end Python workflow** to:

1. Download **MODIS-Aqua Level-3** daily products (**SST**, **chlor_a**, **nFLH**) from NASA OBPG/OB.DAAC.
2. Download **Copernicus Marine** daily products (**SST**, **Chl-a**) via the Copernicus Marine Toolbox.
3. Crop everything to your study area:

**BBox (EPSG:4326):** lon = -64 to -51, lat = -57 to -47

4. Harmonize grids and compute comparison metrics (bias, RMSE, correlation, coverage).

Notes:
- OB.DAAC file search returns **global Level-3 grids**; we crop locally.
- Copernicus Marine supports **server-side subsetting** by bbox/time.
- nFLH is treated as **MODIS-only** (Copernicus Marine products typically do not expose nFLH as a direct variable).

References:
- Copernicus Marine Toolbox subset/describe: https://toolbox-docs.marine.copernicus.eu/  
- Copernicus Marine Toolbox API subset article: https://help.marine.copernicus.eu/en/articles/8283072-copernicus-marine-toolbox-api-subset  
- OBPG nFLH ATBD: https://oceancolor.gsfc.nasa.gov/resources/atbd/nflh/  


## 0) Environment setup

Install (typical):

```bash
pip install requests xarray netcdf4 numpy pandas scipy
pip install copernicusmarine
```

For reprojection / conservative regridding you may also install `xesmf`, but this notebook uses `xarray.interp()` by default.


In [ ]:
from __future__ import annotations

import os
import re
import math
import json
import time
import netrc
import hashlib
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import requests
import xarray as xr
from scipy.stats import pearsonr

from pathlib import Path

In [ ]:
@dataclass(frozen=True)
class BBox:
    xmin: float
    ymin: float
    xmax: float
    ymax: float


BBOX_FI = BBox(xmin=-64.0, ymin=-57.0, xmax=-51.0, ymax=-47.0)

# Choose a date range for the daily comparison.
# Use ISO format YYYY-MM-DD.
DATE_START = "2024-01-01"
DATE_END = "2024-01-07"

# Output folders
OUT_DIR = Path("../Data").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data will be written to:", OUT_DIR)

MODIS_DIR = OUT_DIR / "modis"
COP_DIR = OUT_DIR / "copernicus"
CROPPED_DIR = OUT_DIR / "cropped"
ANALYSIS_DIR = OUT_DIR / "analysis"

for d in [MODIS_DIR, COP_DIR, CROPPED_DIR, ANALYSIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BBox:", BBOX_FI)
print("Date range:", DATE_START, "to", DATE_END)


## 1) MODIS (OB.DAAC) Level-3 daily download (global) + local crop

We use the OB.DAAC `file_search` endpoint to build URL lists for the products.

- Endpoint documentation is linked from OBPG tooling pages.
- We do **not** hardcode dynamic dataset IDs here.
- Instead, we search by filename pattern and date range.

You may need Earthdata Login credentials to download.
The cleanest approach for scripting is to store them in `~/.netrc`.


In [ ]:
OB_DAAC_FILE_SEARCH = "https://oceandata.sci.gsfc.nasa.gov/file_search"

# Documented: Aqua-MODIS is sensor_id=7 in the OB.DAAC file_search service.
# If this ever changes, you'll see it immediately because file_search will return empty results.
AQUA_MODIS_SENSOR_ID = "7"


def _earthdata_auth_from_netrc(machine: str = "urs.earthdata.nasa.gov") -> Optional[tuple[str, str]]:
    """Return (login, password) from ~/.netrc if available; else None."""
    try:
        creds = netrc.netrc().authenticators(machine)
    except (FileNotFoundError, netrc.NetrcParseError):
        return None
    if creds is None:
        return None
    login, _, password = creds
    if not login or not password:
        return None
    return login, password


def modis_file_search(
    search_pattern: str,
    start_date: str,
    end_date: str,
    dtype: str = "L3m",
    sensor_id: str = AQUA_MODIS_SENSOR_ID,
) -> list[str]:
    """Query OB.DAAC file_search and return a list of download URLs (one per file).

    Parameters:
        search_pattern: Wildcard pattern for filenames.
        start_date/end_date: ISO dates YYYY-MM-DD.
        dtype: OB.DAAC data type; for Level-3 mapped products commonly 'L3m'.
        sensor_id: Aqua-MODIS sensor id.
    """
    sdate = f"{start_date} 00:00:00"
    edate = f"{end_date} 23:59:59"
    params = {
        "results_as_file": "1",
        "sensor_id": sensor_id,
        "dtype": dtype,
        "sdate": sdate,
        "edate": edate,
        "addurl": "1",
        "format": "txt",
        "search": search_pattern,
    }
    resp = requests.get(OB_DAAC_FILE_SEARCH, params=params, timeout=60)
    resp.raise_for_status()
    lines = [ln.strip() for ln in resp.text.splitlines() if ln.strip()]
    # The response is plain text URLs (when addurl=1).
    return lines


def download_file(url: str, out_path: Path, overwrite: bool = False) -> Path:
    """Download a file (Earthdata auth via ~/.netrc if needed)."""
    if out_path.exists() and not overwrite:
        return out_path

    auth = _earthdata_auth_from_netrc()
    session = requests.Session()
    if auth is not None:
        session.auth = auth

    with session.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return out_path


### 1.1 Define MODIS filename patterns

Because OB.DAAC filenames are specific, we keep patterns **user-editable**.

Start by running the next cell and inspecting a few returned URLs.
If any list is empty, adjust the pattern(s) until you get matches.


In [ ]:
# Edit these patterns if needed based on what file_search returns.
# The goal is to find daily Level-3 mapped NetCDF files at ~1 km.

MODIS_PATTERNS_DAILY_1KM = {
    "sst": "*L3m*DAY*sst*1km*.nc*",
    "chlor_a": "*L3m*DAY*chlor_a*1km*.nc*",
    "nflh": "*L3m*DAY*nflh*1km*.nc*",
}

modis_url_lists = {}
for var_name, pattern in MODIS_PATTERNS_DAILY_1KM.items():
    urls = modis_file_search(pattern, DATE_START, DATE_END)
    modis_url_lists[var_name] = urls
    print(var_name, "matches:", len(urls))
    for u in urls[:3]:
        print("  ", u)


### 1.2 Download MODIS files

This will download the returned URLs into `data/modis/`.


In [ ]:
def filename_from_url(url: str) -> str:
    return url.rstrip("/").split("/")[-1]


downloaded_modis = {"sst": [], "chlor_a": [], "nflh": []}

for var_name, urls in modis_url_lists.items():
    for url in urls:
        fn = filename_from_url(url)
        out_path = MODIS_DIR / fn
        p = download_file(url, out_path)
        downloaded_modis[var_name].append(p)

for var_name, paths in downloaded_modis.items():
    print(var_name, "downloaded:", len(paths))


### 1.3 Crop MODIS files to the study bbox

We crop using xarray by selecting lon/lat slices.
If your file uses different coordinate names (e.g., `longitude`/`latitude`), update `guess_lon_lat_names()`.


In [ ]:
def guess_lon_lat_names(ds: xr.Dataset) -> tuple[str, str]:
    """Try common lon/lat coordinate names."""
    candidates = [
        ("lon", "lat"),
        ("longitude", "latitude"),
        ("LON", "LAT"),
    ]
    for lon_name, lat_name in candidates:
        if lon_name in ds.coords and lat_name in ds.coords:
            return lon_name, lat_name
        if lon_name in ds.variables and lat_name in ds.variables:
            return lon_name, lat_name
    raise KeyError("Could not find lon/lat coordinate names in dataset")


def crop_to_bbox(ds: xr.Dataset, bbox: BBox) -> xr.Dataset:
    lon_name, lat_name = guess_lon_lat_names(ds)
    # Ensure increasing slices. xarray handles slicing even if coordinates are descending,
    # but we keep it explicit.
    lon_slice = slice(bbox.xmin, bbox.xmax)
    lat_slice = slice(bbox.ymin, bbox.ymax)
    return ds.sel({lon_name: lon_slice, lat_name: lat_slice})


def crop_file(in_path: Path, out_path: Path, bbox: BBox) -> Path:
    with xr.open_dataset(in_path, decode_times=True) as ds:
        cropped = crop_to_bbox(ds, bbox)
        cropped.to_netcdf(out_path)
    return out_path


cropped_modis = {"sst": [], "chlor_a": [], "nflh": []}
for var_name, paths in downloaded_modis.items():
    for in_path in paths:
        out_path = CROPPED_DIR / f"modis_{in_path.name}"
        crop_file(in_path, out_path, BBOX_FI)
        cropped_modis[var_name].append(out_path)

for var_name, paths in cropped_modis.items():
    print(var_name, "cropped:", len(paths))


## 2) Copernicus Marine: server-side subset download (daily SST + Chl-a)

We use the Copernicus Marine Toolbox (`copernicusmarine`) to:
- **describe** datasets to discover the exact variable names
- **subset** by bbox and time range

You must provide dataset IDs appropriate for **daily SST** and **daily chlorophyll-a**.

This notebook intentionally **does not guess** dataset IDs or variable names.
Instead, it helps you discover them via `describe()`.


In [ ]:
try:
    import copernicusmarine  # type: ignore
except ImportError as exc:
    raise ImportError(
        "Missing copernicusmarine. Install with: pip install copernicusmarine"
    ) from exc


# Set these explicitly after discovering them via describe().
COP_SST_DATASET_ID = None  # e.g. an SST daily dataset
COP_CHL_DATASET_ID = None  # e.g. an ocean color daily chlorophyll dataset

print("copernicusmarine version:", getattr(copernicusmarine, "__version__", "unknown"))


### 2.1 Discover datasets and variable names with `describe()`

Run `copernicusmarine.describe()` to explore the catalogue.
This can be large; use filters if needed.


In [ ]:
# This may take a bit depending on your connection.
# If you prefer, you can use CLI outside the notebook:
#   copernicusmarine describe
# and then paste dataset IDs here.

catalogue = copernicusmarine.describe()
print(type(catalogue))
print("Catalogue loaded. Use catalogue.search(...) or browse its content.")


### 2.2 Helper: print variables for a dataset_id

Once you have a dataset_id, run the next cell to list available variables.


In [ ]:
def print_dataset_variables(dataset_id: str) -> None:
    ds_meta = copernicusmarine.describe(dataset_id=dataset_id)
    # The returned object is a catalogue-like object; convert to dict when possible.
    meta_dict = ds_meta.to_dict() if hasattr(ds_meta, "to_dict") else ds_meta
    vars_list = meta_dict.get("datasets", [{}])[0].get("variables", [])
    short_names = [v.get("short_name") for v in vars_list if isinstance(v, dict)]
    print("dataset_id:", dataset_id)
    print("variables (short_name):")
    for name in short_names:
        print("  -", name)


# Example usage (uncomment once you set the dataset IDs):
# print_dataset_variables(COP_SST_DATASET_ID)
# print_dataset_variables(COP_CHL_DATASET_ID)


### 2.3 Subset download (bbox + time)

Once you set dataset IDs and pick variable names from `print_dataset_variables()`, run this.


In [ ]:
def ensure_not_none(value: Optional[str], name: str) -> str:
    if value is None or not str(value).strip():
        raise ValueError(f"{name} must be set (not None).")
    return str(value)


def cop_subset_download(
    dataset_id: str,
    variables: list[str],
    out_file: Path,
    bbox: BBox,
    start_date: str,
    end_date: str,
) -> Path:
    dataset_id = ensure_not_none(dataset_id, "dataset_id")
    if not variables:
        raise ValueError("variables must be a non-empty list")

    # Copernicus Marine Toolbox API subset
    # Uses bbox/time constraints.
    copernicusmarine.subset(
        dataset_id=dataset_id,
        variables=variables,
        minimum_longitude=bbox.xmin,
        maximum_longitude=bbox.xmax,
        minimum_latitude=bbox.ymin,
        maximum_latitude=bbox.ymax,
        start_datetime=f"{start_date}T00:00:00",
        end_datetime=f"{end_date}T23:59:59",
        output_filename=str(out_file),
    )
    return out_file


# After setting dataset IDs and variables, do something like:
# COP_SST_VAR = ["analysed_sst"]  # must come from print_dataset_variables()
# COP_CHL_VAR = ["CHL"]          # must come from print_dataset_variables()
#
# cop_sst_path = cop_subset_download(
#     dataset_id=COP_SST_DATASET_ID,
#     variables=COP_SST_VAR,
#     out_file=COP_DIR / "cop_sst_daily_fi.nc",
#     bbox=BBOX_FI,
#     start_date=DATE_START,
#     end_date=DATE_END,
# )
#
# cop_chl_path = cop_subset_download(
#     dataset_id=COP_CHL_DATASET_ID,
#     variables=COP_CHL_VAR,
#     out_file=COP_DIR / "cop_chl_daily_fi.nc",
#     bbox=BBOX_FI,
#     start_date=DATE_START,
#     end_date=DATE_END,
# )
#
# print(cop_sst_path)
# print(cop_chl_path)


## 3) Harmonize grids and compare (daily)

At this stage you should have:
- MODIS daily cropped SST and chlor_a (many files, typically one per day per variable)
- Copernicus subset daily SST and chlorophyll (one file with a time dimension)

This section provides utilities to:
- open MODIS daily files into a time-indexed xarray Dataset
- regrid Copernicus to MODIS grid (or vice versa)
- compute bias, RMSE, correlation, and coverage


In [ ]:
DATE_RE = re.compile(r"A(\d{4})(\d{3})")


def doy_to_date(year: int, doy: int) -> datetime:
    return datetime(year, 1, 1) + pd.Timedelta(days=doy - 1)


def infer_date_from_modis_filename(name: str) -> datetime:
    """Extract date from typical OBPG MODIS filenames containing AYYYYDDD."""
    m = DATE_RE.search(name)
    if m is None:
        raise ValueError(f"Could not infer date from filename: {name}")
    year = int(m.group(1))
    doy = int(m.group(2))
    return doy_to_date(year, doy)


def open_modis_daily_stack(paths: list[Path], var_name_hint: str) -> xr.DataArray:
    """Open many single-day MODIS L3 files into a time-stacked DataArray.

    This function assumes each file contains exactly one main data variable.
    If your files contain multiple variables, adjust selection logic below.
    """
    arrays = []
    times = []
    for p in sorted(paths):
        dt = infer_date_from_modis_filename(p.name)
        with xr.open_dataset(p, decode_times=True) as ds:
            data_vars = list(ds.data_vars)
            if not data_vars:
                raise ValueError(f"No data variables found in {p}")
            # Choose the first data var by default.
            da = ds[data_vars[0]].load()
        arrays.append(da)
        times.append(np.datetime64(dt.date()))

    stacked = xr.concat(arrays, dim=xr.DataArray(times, dims="time", name="time"))
    stacked.name = var_name_hint
    return stacked


# Example usage (once you have cropped_modis['sst'] and cropped_modis['chlor_a']):
modis_sst_da = open_modis_daily_stack(cropped_modis["sst"], "modis_sst")
modis_chl_da = open_modis_daily_stack(cropped_modis["chlor_a"], "modis_chlor_a")

print(modis_sst_da)
print(modis_chl_da)


In [ ]:
def align_time(a: xr.DataArray, b: xr.DataArray) -> tuple[xr.DataArray, xr.DataArray]:
    """Align on intersecting time coordinate."""
    common = np.intersect1d(a["time"].values, b["time"].values)
    a2 = a.sel(time=common)
    b2 = b.sel(time=common)
    return a2, b2


def regrid_to_target_grid(
    source: xr.DataArray,
    target: xr.DataArray,
) -> xr.DataArray:
    """Regrid `source` to `target` lon/lat grid using xarray.interp.

    This is bilinear interpolation in lon/lat space.
    """
    src = source
    tgt = target
    lon_name, lat_name = guess_lon_lat_names(tgt.to_dataset(name="tgt"))
    return src.interp({lon_name: tgt[lon_name], lat_name: tgt[lat_name]})


def metrics(a: xr.DataArray, b: xr.DataArray) -> pd.DataFrame:
    """Compute daily summary metrics between a and b (same grid/time)."""
    rows = []
    for t in a["time"].values:
        aa = a.sel(time=t)
        bb = b.sel(time=t)
        mask = np.isfinite(aa.values) & np.isfinite(bb.values)
        n = int(mask.sum())
        if n == 0:
            rows.append(
                {
                    "time": pd.to_datetime(str(t)),
                    "n": 0,
                    "coverage": 0.0,
                    "bias": np.nan,
                    "rmse": np.nan,
                    "corr": np.nan,
                }
            )
            continue
        diff = (bb.values - aa.values)[mask]
        bias = float(np.mean(diff))
        rmse = float(np.sqrt(np.mean(diff ** 2)))
        x = aa.values[mask].astype(float)
        y = bb.values[mask].astype(float)
        corr = float(pearsonr(x, y)[0]) if n >= 2 else np.nan
        coverage = float(n / aa.values.size)
        rows.append(
            {
                "time": pd.to_datetime(str(t)),
                "n": n,
                "coverage": coverage,
                "bias": bias,
                "rmse": rmse,
                "corr": corr,
            }
        )
    return pd.DataFrame(rows).sort_values("time")


### 3.1 Load Copernicus files and compare

After you download Copernicus SST/chl files, open them here.

Important: we do not guess the Copernicus variable names.
You must select them from the dataset metadata.


In [ ]:
# Update these paths after running cop_subset_download.
COP_SST_FILE = COP_DIR / "cop_sst_daily_fi.nc"
COP_CHL_FILE = COP_DIR / "cop_chl_daily_fi.nc"

# Update these variable names after inspecting Copernicus dataset variables.
COP_SST_VAR_NAME = None
COP_CHL_VAR_NAME = None


def open_copernicus_da(nc_path: Path, var_name: str) -> xr.DataArray:
    if not nc_path.exists():
        raise FileNotFoundError(nc_path)
    if not var_name:
        raise ValueError("Copernicus variable name must be set")
    ds = xr.open_dataset(nc_path, decode_times=True)
    if var_name not in ds.data_vars:
        raise KeyError(f"{var_name} not found in {nc_path}; available: {list(ds.data_vars)}")
    return ds[var_name]


# Example usage (uncomment once COP_*_VAR_NAME are set):
# cop_sst_da = open_copernicus_da(COP_SST_FILE, COP_SST_VAR_NAME)
# cop_chl_da = open_copernicus_da(COP_CHL_FILE, COP_CHL_VAR_NAME)
#
# # Align times with MODIS
# modis_sst_t, cop_sst_t = align_time(modis_sst_da, cop_sst_da)
# modis_chl_t, cop_chl_t = align_time(modis_chl_da, cop_chl_da)
#
# # Regrid Copernicus to MODIS grid
# cop_sst_on_modis = regrid_to_target_grid(cop_sst_t, modis_sst_t)
# cop_chl_on_modis = regrid_to_target_grid(cop_chl_t, modis_chl_t)
#
# # Metrics (Copernicus - MODIS)
# sst_metrics = metrics(modis_sst_t, cop_sst_on_modis)
# chl_metrics = metrics(modis_chl_t, cop_chl_on_modis)
#
# display(sst_metrics)
# display(chl_metrics)


## 4) MODIS-only nFLH

You already downloaded and cropped MODIS nFLH above. Use it in your downstream modeling alongside whichever SST/chl choice you decide.


In [ ]:
modis_nflh_da = open_modis_daily_stack(cropped_modis["nflh"], "modis_nflh")
print(modis_nflh_da)


## 5) Save comparison outputs

When you compute `sst_metrics` and `chl_metrics`, save them as CSV for reporting.


In [ ]:
# Example:
# sst_metrics.to_csv(ANALYSIS_DIR / "sst_modis_vs_copernicus_daily_metrics.csv", index=False)
# chl_metrics.to_csv(ANALYSIS_DIR / "chl_modis_vs_copernicus_daily_metrics.csv", index=False)
# print("Saved metrics to:", ANALYSIS_DIR)
